# Repository edition: 01_controlled_and_validation

Run from this repository in your existing GPU environment. Source is in `../suites/01_controlled_and_validation/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Full PopQA revision experiments — real models, real generations

**Select Python (Three Ways), then Run All. No cohort files or placeholder paths are needed.**

This notebook downloads PopQA and the selected pretrained model, builds a new factual-recall cohort, and runs the main revision analyses plus the enabled baseline and internal-intervention studies. There is no smoke mode and no synthetic fallback for the natural-task analysis.

Defaults: Llama 3.2 1B; up to 100 evaluation prompts per relation across six relations; up to 20 successful calibration examples per relation; 200 calibration resamples. Extra experiments are enabled below. This is a long-running GPU job; attribution and intervention loops can dominate runtime.

**Scope:** a new, restricted factual-recall study, not exact reproduction of the original paper. The primary cohort includes full-answer successes at the canonical initial token and unambiguous initial-token failures. Later-token failures and alternative-alias initials are audited and excluded. Automatic answer scoring needs manual review before publication. Extraction/composition and the original multi-model cohorts are not recreated here.

The independent answer inventory, calibration cohort and evaluation cohort use disjoint subject groups. Toy controlled experiments are clearly separated from natural PopQA results.

Dataset: [PopQA](https://huggingface.co/datasets/akariasai/PopQA).


In [ ]:
# 1. Full-run settings. These defaults run real experiments.
MODELS = ["meta-llama/Llama-3.2-1B"]
# Add other supported models if desired, e.g. "Qwen/Qwen2.5-3B".
RUN_NAME = "full_popqa_v1"         # Keep this name to resume; change it for a new configuration.
SEED = 0
EVALUATION_PER_RELATION = 100      # Up to 600 prompts; availability/exclusions are reported.
CALIBRATION_SUCCESSES_PER_RELATION = 20
MAX_CALIBRATION_ATTEMPTS_PER_RELATION = 250
BOOTSTRAP_REPEATS = 200

RUN_NATURAL_BASELINES = True
RANKING_EXAMPLES = 60
RANKING_CANDIDATES = 32            # Uniform candidate subset; not a full-network oracle.
IG_STEPS = 16
RUN_INTERNAL_REPAIRS = True       # Only verified successful donors are retained.
INTERNAL_LAYER_FRACTIONS = [0.0, 0.25, 0.5, 0.75]
RUN_CONTROLLED_BASELINES = True
RUN_STAGE_CONTROLS = True
TOY_SEEDS = 12
TOY_TRAINING_STEPS = 2500
TOY_EXAMPLES_PER_CONDITION = 60


## 2. Load code and check the environment
All code is bundled here. This cell installs the Python dependencies into the selected kernel environment, preserves an already-compatible PyTorch installation, and requires an accessible CUDA GPU. It does not install NVIDIA drivers.


In [ ]:
import base64, io, json, os, subprocess, sys, zipfile, hashlib, time, shutil
from pathlib import Path

BASE = Path.cwd().resolve()
CODE = BASE / ("threeways_full_code_" + hashlib.sha256(base64.b64decode(_source_bundle("01_controlled_and_validation"))).hexdigest()[:12])
CODE.mkdir(exist_ok=True)
_BUNDLE = _source_bundle('01_controlled_and_validation')
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_BUNDLE))) as archive:
    for member in archive.infolist():
        destination = (CODE / member.filename).resolve()
        if CODE.resolve() not in destination.parents:
            raise ValueError("Invalid bundled path")
        if not destination.exists(): archive.extract(member, CODE)
del _BUNDLE

# Use the existing environment; dependencies are documented in requirements.txt.
import torch
import transformers
assert transformers.__version__ == "4.48.3", "Restart the kernel after installation, then Run All again."
assert torch.cuda.is_available(), "CUDA is unavailable. Select the Three Ways kernel and check your GPU-enabled PyTorch."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Python:", sys.executable)

ROOT = BASE / "threeways_full_results" / RUN_NAME
ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = {"models": MODELS, "seed": SEED, "evaluation_per_relation": EVALUATION_PER_RELATION,
    "calibration_successes_per_relation": CALIBRATION_SUCCESSES_PER_RELATION,
    "calibration_attempts_per_relation": MAX_CALIBRATION_ATTEMPTS_PER_RELATION,
    "bootstrap_repeats": BOOTSTRAP_REPEATS, "natural_baselines": RUN_NATURAL_BASELINES,
    "ranking_examples": RANKING_EXAMPLES, "ranking_candidates": RANKING_CANDIDATES,
    "ig_steps": IG_STEPS, "internal_repairs": RUN_INTERNAL_REPAIRS,
    "internal_layer_fractions": INTERNAL_LAYER_FRACTIONS,
    "controlled_baselines": RUN_CONTROLLED_BASELINES, "stage_controls": RUN_STAGE_CONTROLS,
    "toy_seeds": TOY_SEEDS, "toy_training_steps": TOY_TRAINING_STEPS,
    "toy_examples": TOY_EXAMPLES_PER_CONDITION, "code_bundle": "1032f37187cb"}
config_file = ROOT / "configuration.json"
if config_file.exists() and json.loads(config_file.read_text()) != CONFIG:
    raise ValueError("This RUN_NAME has another configuration. Choose a new RUN_NAME to avoid mixing results.")
config_file.write_text(json.dumps(CONFIG, indent=2))
print("Persistent result directory:", ROOT)

def run_module(module, *arguments):
    command = [sys.executable, "-u", "-m", module, *map(str, arguments)]
    env = os.environ.copy()
    env.setdefault("OMP_NUM_THREADS", "1")
    env["TOKENIZERS_PARALLELISM"] = "false"
    log_file = ROOT / "execution.log"
    print("\nRunning", module, flush=True)
    with log_file.open("a") as log:
        log.write("\nCOMMAND " + json.dumps(command) + "\n")
        with subprocess.Popen(command, cwd=CODE, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"{module} failed (exit {status}). Read the error above. Fix the cause and rerun this cell; completed records are retained.")

STATUS = []
def note(stage, model, message):
    STATUS.append(dict(stage=stage, model=model, message=message))
    (ROOT / "stage_status.json").write_text(json.dumps(STATUS, indent=2))
    print(message)


## 3. Download the data/model snapshots
If requested, enter a Hugging Face read token in the hidden input. Your account must have access to the selected gated model. For an ungated alternative, set `MODELS = ["Qwen/Qwen2.5-3B"]` and use a new `RUN_NAME`.

The notebook pins model and dataset commit IDs on the first run and reuses them on resume.


In [ ]:
from huggingface_hub import HfApi, get_token, login, snapshot_download
from getpass import getpass
if get_token() is None and any(name.startswith("meta-llama/") for name in MODELS):
    login(token=getpass("Hugging Face read token (hidden): "), add_to_git_credential=False)
api = HfApi()
pins_file = ROOT / "snapshots.json"
if pins_file.exists():
    PINS = json.loads(pins_file.read_text())
else:
    PINS = {"dataset": api.dataset_info("akariasai/PopQA").sha,
            "models": {name: api.model_info(name).sha for name in MODELS}}
    pins_file.write_text(json.dumps(PINS, indent=2))

MODEL_PATHS = {}
for name in MODELS:
    MODEL_PATHS[name] = snapshot_download(repo_id=name, revision=PINS["models"][name],
        allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.tiktoken"])
    print(name, "→", PINS["models"][name])
print("PopQA snapshot:", PINS["dataset"])

def model_dir(name):
    return ROOT / name.replace("/", "__")


## 4. Build real cohorts, independent decoys and verified donors
For each model, this cell runs real greedy generations. It saves every screened prediction and its inclusion/exclusion reason, creates the calibration/evaluation files, and constructs independently held-out type-matched decoys.

It stops rather than inventing data if fewer than 10 independent calibration-success or evaluation-failure groups are available. Inspect `coverage.json` and `annotation_audit.csv` if this happens. Calibration candidate sampling stops at the declared success target; evaluation samples are chosen before observing model outcomes.


In [ ]:
import pandas as pd
for name in MODELS:
    directory = model_dir(name)
    run_module("revision.build_real", "--model", MODEL_PATHS[name],
        "--dataset-revision", PINS["dataset"], "--out", directory / "data", "--device", "cuda",
        "--seed", SEED, "--evaluation-per-relation", EVALUATION_PER_RELATION,
        "--calibration-successes-per-relation", CALIBRATION_SUCCESSES_PER_RELATION,
        "--calibration-attempts-per-relation", MAX_CALIBRATION_ATTEMPTS_PER_RELATION,
        "--ranking-examples", RANKING_EXAMPLES)
    print("\nCoverage:", name)
    display(pd.DataFrame(json.loads((directory / "data/coverage.json").read_text())))
    note("cohort", name, "Real cohort and decoy files saved.")


## 5. Run the full main analysis
Measures both transport definitions, all five source-span variants and layer-cut checks; runs 200 resamples by default; varies decoy size, threshold and readability window; tests leave-relation-out calibration; evaluates readout repairs and held-out utility.

The main reference uses two consecutive layers, 10th-percentile calibration and five random decoys. Decoy-size conditions with insufficient inventory are explicitly excluded. The figures show a declared slice; use the full CSVs for the other settings. Uncertainty from calibration resampling is conditional on the evaluation cohort.


In [ ]:
for name in MODELS:
    directory = model_dir(name)
    run_module("revision.extract", "--model", MODEL_PATHS[name],
        "--cohort", directory / "data/cohort.jsonl", "--decoy-pool", directory / "data/decoys.json",
        "--out", directory / "measured", "--device", "cuda", "--seed", SEED)
    run_module("revision.robustness", "--measurements", directory / "measured",
        "--out", directory / "robustness", "--repeats", BOOTSTRAP_REPEATS,
        "--sizes", 20, 40, 60, "--decoy-sizes", 5, 10, 20,
        "--consecutive", 1, 2, 3, "--transfer", "--seed", SEED)
    features = directory / "robustness/reference_features.csv"
    data = pd.read_csv(features, dtype={"group_id": str})
    if data.group_id.nunique() >= 5:
        for arm in ["answer_up", "combined", "alt_down", "fixed_answer_1"]:
            run_module("revision.predict", "--features", features,
                "--outcomes", directory / "robustness/repairs.csv", "--arm", arm,
                "--out", directory / ("utility_" + arm), "--seed", SEED)
    else:
        note("utility", name, "Not estimated: fewer than five eligible independent groups after numerical/decoy exclusions.")
    run_module("revision.report", "--robustness", directory / "robustness/summary.csv",
        "--out", directory / "figures", "--seed", SEED)
    note("main_analysis", name, "Main real-data analysis completed.")


## 6. Natural-task intervention benchmarks
Uses up to 60 prespecified failed examples, spread across relations. Compares Affine edge sums, gradient × activation, the derivative of the matched renormalized removal, and candidate-specific integrated gradients with exact removal effects.

The default 32 candidates are sampled without consulting attribution scores. The oracle is only over those candidates. Integrated gradients here is an expensive candidate-specific reference integrator, **not the published EAP-IG or AtP\*** algorithm.


In [ ]:
if RUN_NATURAL_BASELINES:
    for name in MODELS:
        directory = model_dir(name)
        run_module("revision.natural_ranking", "--model", MODEL_PATHS[name],
            "--cohort", directory / "data/ranking_cohort.jsonl",
            "--out", directory / "natural_ranking", "--device", "cuda",
            "--candidates", RANKING_CANDIDATES, "--ig-steps", IG_STEPS, "--seed", SEED)
        run_module("revision.report", "--candidates", directory / "natural_ranking/candidates.csv",
            "--out", directory / "natural_ranking/report", "--seed", SEED)
        note("natural_ranking", name, "Natural-task intervention rankings completed.")
else:
    note("natural_ranking", "all", "Disabled explicitly in settings.")


## 7. Internal repairs and matched controls
Patches source-span residual states using verified successful context donors. Random-vector controls and available wrong-answer donors are matched in displacement norm. All diagnostic categories receive the same repair families. Four layer fractions are prespecified; the notebook does not select a best layer using the outcomes.

Donor availability is selective and reported. These tests assess repair associations, not unique causal stage identification.


In [ ]:
from transformers import AutoConfig
if RUN_INTERNAL_REPAIRS:
    for name in MODELS:
        directory = model_dir(name)
        audit = json.loads((directory / "data/donor_coverage.json").read_text())
        available = sum(row["eligible"] for row in audit)
        print(name, "verified donors:", available, "of", len(audit))
        if not available:
            note("internal_repairs", name, "Not estimated: no verified successful aligned donors. See donor_coverage.json.")
            continue
        depth = AutoConfig.from_pretrained(MODEL_PATHS[name]).num_hidden_layers
        layers = sorted({min(depth-1, int(fraction*depth)) for fraction in INTERNAL_LAYER_FRACTIONS})
        run_module("revision.internal", "--model", MODEL_PATHS[name],
            "--cohort", directory / "data/internal_cohort.jsonl",
            "--out", directory / "internal", "--device", "cuda",
            "--layers", *layers, "--alphas", .25, .5, 1., "--ig-steps", IG_STEPS, "--seed", SEED)
        observed = pd.read_csv(directory / "internal/internal_repairs.csv", dtype={"id": str})
        labels = pd.read_csv(directory / "robustness/reference_features.csv", dtype={"id": str})
        if set(observed.id) & set(labels.id):
            run_module("revision.internal_report", "--features", directory / "robustness/reference_features.csv",
                "--repairs", directory / "internal/internal_repairs.csv",
                "--out", directory / "internal/report", "--seed", SEED)
        else:
            note("internal_report", name, "No overlap with numerically eligible reference-labelled examples; raw repairs retained.")
        note("internal_repairs", name, "Verified-donor repair experiments completed; inspect coverage and small subgroups.")
else:
    note("internal_repairs", "all", "Disabled explicitly in settings.")


## 8. Full trained controlled experiments
These are deliberately controlled toy retrieval models, trained for 2,500 updates per seed by default—not the two-update smoke configuration. They are separate from the pretrained-model PopQA study.

The easy/decoy/competing/hard conditions share paired prompts. Known-operation tests alter source tokens, suppress transport, or add competitor output bias. Ineffective interventions and insufficient-success coverage are retained. Reusing the same training checkpoints across both controlled suites saves training time; these are not independently trained repetitions.


In [ ]:
if RUN_CONTROLLED_BASELINES:
    run_module("revision.controlled", "--out", ROOT / "controlled", "--device", "cuda",
        "--seeds", TOY_SEEDS, "--steps", TOY_TRAINING_STEPS,
        "--examples", TOY_EXAMPLES_PER_CONDITION, "--ig-steps", IG_STEPS)
    run_module("revision.report", "--candidates", ROOT / "controlled/candidates.csv",
        "--out", ROOT / "controlled/report", "--seed", SEED)
    note("controlled_baselines", "trained_toy", "Full trained controlled baseline suite completed.")
if RUN_STAGE_CONTROLS:
    if RUN_CONTROLLED_BASELINES:
        target = ROOT / "stage_controls/checkpoints"
        target.mkdir(parents=True, exist_ok=True)
        for source in (ROOT / "controlled/checkpoints").glob("*.pt"):
            if not (target/source.name).exists(): shutil.copy2(source, target/source.name)
    run_module("revision.stage_controls", "--out", ROOT / "stage_controls", "--device", "cuda",
        "--seeds", TOY_SEEDS, "--steps", TOY_TRAINING_STEPS, "--calibration", 100, "--examples", 100)
    note("stage_controls", "trained_toy", "Known-operation controls completed; check coverage.csv for each seed.")


## 9. View results and download the analysis
All raw measurements and checkpoints stay on the Lambda disk. The download ZIP contains the analysis tables, figures, cohort definitions, scoring audits, manifests and logs; model weights and large measurement caches are excluded.

Before using results in the paper, inspect the scoring audit, numerical convergence, coverage, label stability and negative results. A completed run does not establish the causal interpretation of the labels.


In [ ]:
from IPython.display import display, SVG, FileLink
for name in MODELS:
    directory = model_dir(name)
    print("\n", name)
    metrics = directory / "utility_answer_up/metrics.csv"
    if metrics.exists(): display(pd.read_csv(metrics))
    figure = directory / "figures/robustness.svg"
    if figure.exists(): display(SVG(filename=str(figure)))
    numerical = directory / "natural_ranking/report/numerical_checks.csv"
    if numerical.exists(): display(pd.read_csv(numerical))
display(pd.DataFrame(STATUS))

archive_path = BASE / (RUN_NAME + "_results.zip")
excluded_dirs = {"measured", "checkpoints", "examples", "screened", "donors", "mpl-cache"}
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in ROOT.rglob("*"):
        if file.is_file() and not excluded_dirs.intersection(file.relative_to(ROOT).parts):
            archive.write(file, file.relative_to(ROOT))
    archive.write(CODE / "revision/build_real.py", "cohort_builder_source.py")
display(FileLink(archive_path.name, result_html_prefix="Download full analysis: "))
print("Persistent results:", ROOT)
print("Resume after interruption: keep the same settings and RUN_NAME, then rerun the interrupted cell.")


### Validation and limits
The code and cohort-builder logic were tested locally (including a mocked dataset/model integration test); the full pretrained-model GPU workload has not been executed here. There is no claim of measured paper performance in this notebook.

This notebook adds real-data construction to the prior experiment suite. Exact published AtP*/EAP-IG integration, the original cohorts, semantic-divergence annotation, and full extraction/composition experiments remain outside its scope. The primary population is explicitly the retained first-token factual-recall cohort.

If any stage stops, read the saved `execution.log`. Do not bypass failed reconstruction checks or silently substitute synthetic data. Change `RUN_NAME` when changing settings to keep experiments separate.
